# Phase 2.3: LSTM Training — Horizon **4 h** (MVP)

## Objectif
Un seul modèle horaire **+1…+4 h** pour alertes shift / conformité (le dual 2 h/4 h est retiré : 2 h sous-performait en test).

## Pipeline (notebook 05)
- Split temporel **par site** | winsor p1–p99 | MinMax sur train
- **Sprint 2** : Huber pondéré + early stopping `val_skill`
- **Sprint 3** : features calendrier (sin/cos heure, jour) — relancer **05** avant **06**

## Sorties
- `model_lstm_4h.h5`, `lstm_4h_metrics.json` (incl. MAE/R² **par polluant**)
- `lstm_4h_skill_report.json` + `lstm_4h_skill_mae_bars.png` (Sprint 1 — skill vs persistance)
- Courbes + 2 figures prédiction (échantillon **médian** + **challenging**)

In [11]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)
sns.set_style("whitegrid")

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config as config_module
import lstm_training as lstm_training_module

importlib.reload(config_module)
importlib.reload(lstm_training_module)

from config import LSTM_ACCEPTANCE, LSTM_CONFIG, N_FEATURES, lstm_input_feature_count
from lstm_training import (
    artifact_paths_4h,
    build_lstm_model,
    build_skill_report,
    clip_predictions,
    compile_lstm_model,
    compare_models_on_test,
    evaluate_forecast,
    evaluate_per_pollutant,
    persistence_baseline,
    pick_plot_sample_idx,
    plot_skill_mae_bars,
    training_callbacks,
)

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
HORIZON = LSTM_CONFIG["horizon_short"]
PATHS = artifact_paths_4h(MODELS_DIR)
tensors_file = DATA_DIR / "lstm_train_val_test.pkl"

print(f"TensorFlow {tf.__version__} | Horizon={HORIZON}h | loss={LSTM_CONFIG['loss']}")

TensorFlow 2.21.0 | Horizon=4h | loss=huber


In [12]:
with open(tensors_file, "rb") as f:
    lstm_data = pickle.load(f)

lookback = lstm_data["lookback"]
if HORIZON not in lstm_data["combined_tensors"]:
    raise KeyError(f"Horizon {HORIZON}h absent — relancer notebook 05 (MVP 4 h seul)")

combined = lstm_data["combined_tensors"][HORIZON]
X_train, y_train = combined["train"]
X_val, y_val = combined["val"]
X_test, y_test = combined["test"]
test_site_ids = combined.get("test_site_ids")
pollutants = lstm_data["pollutant_names"]
N_INPUT = int(X_train.shape[2])
N_OUTPUT = int(y_train.shape[2])
LSTM_CONFIG["n_output_features"] = N_OUTPUT
print(f"Tenseurs: X {X_train.shape} → y {y_train.shape} | entrée={N_INPUT}, sortie={N_OUTPUT}")
if test_site_ids is None:
    print("⚠ test_site_ids absent — relancer notebook 05 pour eval par site")

for name, arr in [("X_train", X_train), ("y_train", y_train), ("X_val", X_val)]:
    if np.isnan(arr).any() or np.isinf(arr).any():
        raise ValueError(f"{name} contient NaN/Inf — relancer notebook 05")

print(f"X_train {X_train.shape} → y_train {y_train.shape}")
print(f"Lookback={lookback}h | Timestep={lstm_data['timestep_minutes']}min")

Entrée LSTM: 8 features | Sortie: 8 polluants
X_train (124711, 48, 12) → y_train (124711, 4, 8)
Lookback=48h | Timestep=60min


In [13]:
def plot_prediction_grid(y_true, y_pred, sample_idx, title, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, p_idx in zip(axes.flatten(), range(min(4, N_FEATURES))):
        steps = np.arange(1, HORIZON + 1)
        ax.plot(steps, y_true[sample_idx, :, p_idx], "o-", label="Réel")
        ax.plot(steps, y_pred[sample_idx, :, p_idx], "s--", label="Prédit")
        ax.set_title(pollutants[p_idx])
        ax.set_xlabel("Heure (+h)")
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()


# Dimensions depuis les tenseurs (évite décalage metadata / vieux noyau Jupyter)
if "X_train" not in globals():
    with open(tensors_file, "rb") as f:
        lstm_data = pickle.load(f)
    combined = lstm_data["combined_tensors"][HORIZON]
    X_train, y_train = combined["train"]
    X_val, y_val = combined["val"]
    X_test, y_test = combined["test"]
    test_site_ids = combined.get("test_site_ids")
    pollutants = lstm_data["pollutant_names"]
    lookback = lstm_data["lookback"]
N_INPUT = int(X_train.shape[2])
N_OUTPUT = int(y_train.shape[2])
LSTM_CONFIG["n_output_features"] = N_OUTPUT
print(f"Tenseurs: X {X_train.shape} → y {y_train.shape} | modèle {N_INPUT}→{N_OUTPUT}")
if N_INPUT == N_OUTPUT and LSTM_CONFIG.get("use_calendar_features"):
    raise ValueError(
        "Entrée=8 sans calendrier — relancer notebook 05 (use_calendar_features) puis cellule tenseurs"
    )

tf.keras.backend.clear_session()
model = build_lstm_model(lookback, HORIZON, N_INPUT, N_OUTPUT, LSTM_CONFIG)
compile_lstm_model(model, LSTM_CONFIG)
model.summary()
assert model.input_shape[-1] == N_INPUT, (model.input_shape, N_INPUT)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=LSTM_CONFIG["epochs"],
    batch_size=LSTM_CONFIG["batch_size"],
    callbacks=training_callbacks(
        LSTM_CONFIG, PATHS["model"], X_val=X_val, y_val=y_val, horizon=HORIZON
    ),
    verbose=1,
)
print(f"Entraînement terminé — {len(history.history['loss'])} epochs")
if history.history.get("val_skill"):
    print(f"Best val_skill: {max(history.history['val_skill']):.4f}")

n_plots = 3 if history.history.get("val_skill") else 2
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
if n_plots == 2:
    axes = np.array([axes[0], axes[1]])
loss_label = (
    f"weighted {LSTM_CONFIG['loss']}"
    if LSTM_CONFIG.get("use_weighted_loss")
    else LSTM_CONFIG["loss"]
)
axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Val")
axes[0].set_title(f"LSTM {HORIZON}h — Loss ({loss_label})")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history.history["mae"], label="Train")
axes[1].plot(history.history["val_mae"], label="Val")
axes[1].set_title(f"LSTM {HORIZON}h — MAE")
axes[1].legend()
axes[1].grid(alpha=0.3)
if n_plots == 3:
    axes[2].plot(history.history["val_skill"], color="green")
    axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[2].set_title(f"Val skill ({LSTM_CONFIG.get('early_stopping_monitor')})")
    axes[2].set_ylabel("1 − MAE_LSTM/MAE_pers")
    axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PATHS["curves"], dpi=100, bbox_inches="tight")
plt.show()

y_pred = clip_predictions(model.predict(X_test, verbose=0))
y_persist = clip_predictions(persistence_baseline(X_test, HORIZON, N_OUTPUT))
metrics = evaluate_forecast(y_test, y_pred)
per_pollutant = evaluate_per_pollutant(y_test, y_pred, pollutants)
baseline_cmp = compare_models_on_test(y_test, y_pred, y_persist, pollutants)
metrics.update({
    "horizon_steps": HORIZON,
    "horizon_hours": HORIZON,
    "lookback_hours": lookback,
    "timestep_minutes": lstm_data["timestep_minutes"],
    "loss": LSTM_CONFIG["loss"],
    "use_weighted_loss": LSTM_CONFIG.get("use_weighted_loss"),
    "early_stopping_monitor": LSTM_CONFIG.get("early_stopping_monitor"),
    "best_val_skill": max(history.history.get("val_skill", [0])),
    "test_samples": int(len(y_test)),
    "epochs_trained": len(history.history["loss"]),
    "per_pollutant": per_pollutant,
    "baseline_comparison": baseline_cmp,
})
print(f"LSTM  — RMSE={metrics['rmse']:.4f} | MAE={metrics['mae']:.4f} | R²={metrics['r2_score']:.4f}")
bp = baseline_cmp["persistence"]
print(f"Pers. — RMSE={bp['rmse']:.4f} | MAE={bp['mae']:.4f} | R²={bp['r2_score']:.4f}")
print(
    f"ΔMAE={baseline_cmp['delta_mae']:+.4f} (négatif = LSTM meilleur) | "
    f"LSTM gagne MAE: {baseline_cmp['lstm_wins_mae']}"
)
print("\nMAE par pas d'horizon (LSTM vs persistance):")
for step, lstm_h in baseline_cmp["per_horizon_lstm"].items():
    pers_h = baseline_cmp["per_horizon_persistence"][step]
    print(
        f"  {step}: LSTM MAE={lstm_h['mae']:.4f} | Pers. MAE={pers_h['mae']:.4f} | "
        f"Δ={lstm_h['mae'] - pers_h['mae']:+.4f}"
    )
print("\nPar polluant LSTM (MAE / R²):")
for name, m in per_pollutant.items():
    pp = baseline_cmp["per_pollutant_persistence"][name]
    print(
        f"  {name:12s} LSTM MAE={m['mae']:.4f} R²={m['r2_score']:.4f} | "
        f"Pers. MAE={pp['mae']:.4f} R²={pp['r2_score']:.4f}"
    )

idx_median = pick_plot_sample_idx(y_test, "median")
idx_hard = pick_plot_sample_idx(y_test, "challenging")
plot_prediction_grid(
    y_test, y_pred, idx_median,
    f"LSTM {HORIZON}h — échantillon médian ({idx_median})",
    PATHS["predictions"],
)
plot_prediction_grid(
    y_test, y_pred, idx_hard,
    f"LSTM {HORIZON}h — cas difficile ({idx_hard})",
    PATHS["predictions_challenging"],
)

skill_report = build_skill_report(
    y_test, y_pred, y_persist, pollutants,
    site_ids=test_site_ids,
    acceptance=LSTM_ACCEPTANCE,
)
metrics["skill_report_summary"] = {
    "global_skill": skill_report["global"]["skill"],
    "go_deploy": skill_report["acceptance"]["go_deploy"],
    "recommendation": skill_report["acceptance"]["recommendation"],
    "failed_checks": skill_report["acceptance"]["failed_checks"],
}
plot_skill_mae_bars(
    skill_report,
    PATHS["skill_chart"],
    title=f"LSTM {HORIZON}h — skill vs persistance (test)",
)
with open(PATHS["skill_report"], "w", encoding="utf-8") as f:
    json.dump(skill_report, f, indent=2, default=str)

acc = skill_report["acceptance"]
print(f"\n=== Sprint 1 — Skill report ===")
print(f"Skill global: {skill_report['global']['skill']:.4f}")
print(f"go_deploy: {acc['go_deploy']} | {acc['recommendation']}")
if acc["failed_checks"]:
    print(f"Checks échoués: {', '.join(acc['failed_checks'])}")
for c in acc["checks"]:
    status = "OK" if c["passed"] else "FAIL"
    print(f"  [{status}] {c['message']}")

model.save(PATHS["model"])
with open(PATHS["metrics"], "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
with open(PATHS["history"], "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)
print(f"Modèle: {PATHS['model']}")
print(f"Métriques: {PATHS['metrics']}")
print(f"Skill report: {PATHS['skill_report']}")
print(f"Graphique skill: {PATHS['skill_chart']}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 48, 64)         │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 4, 8)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,304 (126.19 KB)

 Trainable params: 32,240 (125.94 KB)

 Non-trainable params: 64 (256.00 B)

Epoch 1/100


ValueError: Exception encountered when calling LSTMCell.call().

[1mDimensions must be equal, but are 12 and 8 for '{{node sequential_1_1/lstm_2_1/lstm_cell_1/MatMul}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](sequential_1_1/lstm_2_1/strided_slice_2, sequential_1_1/lstm_2_1/lstm_cell_1/Cast/ReadVariableOp)' with input shapes: [?,12], [8,256].[0m

Arguments received by LSTMCell.call():
  • inputs=tf.Tensor(shape=(None, 12), dtype=float32)
  • states=('tf.Tensor(shape=(None, 64), dtype=float32)', 'tf.Tensor(shape=(None, 64), dtype=float32)')
  • training=True

## Sprint 1 — Rapport skill (sans ré-entraîner)

Exécuter cette cellule après la cellule d’entraînement, **ou** seule si `model_lstm_4h.h5` existe déjà. Requiert `test_site_ids` dans le `.pkl` (notebook 05 régénéré).

In [ ]:
# Réévaluation skill seule (pas de fit) — autonome si cellules 1–2 non exécutées
import json
import pickle
import sys
from pathlib import Path

import numpy as np
from tensorflow import keras

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config as config_module
import lstm_training as lstm_training_module

importlib.reload(config_module)
importlib.reload(lstm_training_module)

from config import LSTM_ACCEPTANCE, LSTM_CONFIG
from lstm_training import (
    artifact_paths_4h,
    build_skill_report,
    clip_predictions,
    load_lstm_model_for_inference,
    persistence_baseline,
    plot_skill_mae_bars,
)

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
HORIZON = LSTM_CONFIG["horizon_short"]
PATHS = artifact_paths_4h(MODELS_DIR)

try:
    _ = X_test.shape
except NameError:
    with open(DATA_DIR / "lstm_train_val_test.pkl", "rb") as f:
        lstm_data = pickle.load(f)
    combined = lstm_data["combined_tensors"][HORIZON]
    X_test, y_test = combined["test"]
    test_site_ids = combined.get("test_site_ids")
    pollutants = lstm_data["pollutant_names"]
    if test_site_ids is None:
        print("⚠ test_site_ids absent — relancer notebook 05 pour eval par site")

if "y_pred" not in globals() or "y_persist" not in globals():
    if not PATHS["model"].exists():
        raise FileNotFoundError(f"Modèle absent: {PATHS['model']} — exécuter la cellule d'entraînement")
    model = load_lstm_model_for_inference(PATHS["model"])
    y_pred = clip_predictions(model.predict(X_test, verbose=0))
    y_persist = clip_predictions(persistence_baseline(X_test, HORIZON))

site_ids_arg = globals().get("test_site_ids")
skill_report = build_skill_report(
    y_test, y_pred, y_persist, pollutants,
    site_ids=site_ids_arg,
    acceptance=LSTM_ACCEPTANCE,
)
plot_skill_mae_bars(skill_report, PATHS["skill_chart"])
with open(PATHS["skill_report"], "w", encoding="utf-8") as f:
    json.dump(skill_report, f, indent=2, default=str)

acc = skill_report["acceptance"]
print(f"go_deploy={acc['go_deploy']} | {acc['recommendation']}")
print(f"Skill global: {skill_report['global']['skill']:.4f}")
for p, s in skill_report["per_pollutant_skill"].items():
    fb = " (fallback)" if p in LSTM_ACCEPTANCE["allow_fallback_pollutants"] else ""
    print(f"  {p:12s} skill={s:+.4f}{fb}")

if skill_report.get("per_site"):
    sites_sorted = sorted(
        skill_report["per_site"].items(),
        key=lambda x: x[1]["global_skill"],
        reverse=True,
    )
    print("\nTop 5 sites (skill global):")
    for site, m in sites_sorted[:5]:
        print(f"  {site}: skill={m['global_skill']:.4f} n={m['n_samples']}")
    print("Bottom 5 sites:")
    for site, m in sites_sorted[-5:]:
        print(f"  {site}: skill={m['global_skill']:.4f} n={m['n_samples']}")
else:
    print("\n(per_site: relancer notebook 05 puis recharger les tenseurs)")

PATHS["skill_chart"]

go_deploy=True | deploy_with_hybrid_fallback
Skill global: -0.1639
  CO2          skill=+0.1342
  NOX          skill=-3.2655 (fallback)
  SOX          skill=-0.6698 (fallback)
  PM25         skill=-3.1696 (fallback)
  PM10         skill=+0.1419
  COV          skill=-0.4965 (fallback)
  TEMPERATURE  skill=+0.0623
  HUMIDITY     skill=-0.1126 (fallback)

Top 5 sites (skill global):
  48_113_69: skill=0.2328 n=180
  11_1_43: skill=0.2119 n=212
  48_201_1035: skill=0.0738 n=975
  24_5_3001: skill=0.0596 n=436
  Aotizhongxin: skill=0.0060 n=3927
Bottom 5 sites:
  8_45_19: skill=-1.0995 n=195
  49_35_3006: skill=-1.5389 n=285
  49_35_3018: skill=-2.0187 n=652
  6_19_4001: skill=-3.4978 n=296
  48_439_1002: skill=-3.8122 n=170


WindowsPath('C:/Users/melik/Desktop/pollution_monitoring/ia/models/lstm_4h_skill_mae_bars.png')

In [ ]:
# Entraînement exécuté dans la cellule précédente (boucle 2h + 4h).

In [ ]:
# Courbes sauvegardées par horizon dans la cellule d'entraînement.

In [ ]:
# Métriques test affichées dans la boucle d'entraînement (sans MAPE sur [0,1]).

In [ ]:
# Graphiques prédictions par horizon dans la cellule d'entraînement.

In [ ]:
# Modèles et JSON exportés dans la boucle (model_lstm_2h.h5, model_lstm_4h.h5).